# Normal-tissue protein co-representation

In [8]:
import subprocess

print("paths.root:", paths.root)

for args in (
        ["git", "rev-parse", "--show-toplevel"],
        ["git", "rev-parse", "--verify", "HEAD"],
        ["git", "status", "--short"],
):
    result = subprocess.run(
        args, cwd=paths.root, capture_output=True, text=True
    )
    print(f"\n{args}")
    print("return code:", result.returncode)
    print("stdout:", result.stdout.strip())
    print("stderr:", result.stderr.strip())

paths.root: D:\IdeaProjects\CAP_Submission\CAP_Analysis

['git', 'rev-parse', '--show-toplevel']
return code: 128
stdout: 
stderr: fatal: not a git repository (or any of the parent directories): .git

['git', 'rev-parse', '--verify', 'HEAD']
return code: 128
stdout: 
stderr: fatal: not a git repository (or any of the parent directories): .git

['git', 'status', '--short']
return code: 128
stdout: 
stderr: fatal: not a git repository (or any of the parent directories): .git


## Imports


In [1]:
from __future__ import annotations
from pathlib import Path
import hashlib
import itertools
import json


In [2]:
import os
import zipfile
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
import numpy as np


In [3]:
import pandas as pd
import session_info


In [4]:
import logging
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)

## Paths


In [5]:
# Add to your imports
from cap.paths import get_paths
from cap.provenance import (sha256_file, validate_frozen_inputs, write_run_provenance)

paths = get_paths()
paths.validate_base_inputs()
validate_frozen_inputs(paths)

project = paths.root
analysis_dir = paths.run_root

In [6]:
# Resolve the notebook source relative to the repository root.
notebook_path = paths.root / "notebooks" / "003d_Normal_Tissue_Protein.ipynb"

In [7]:
# Resolve and validate inputs before reserving the stage.
data_dir = paths.data_root
data_file = data_dir / "references" / "hpa" / "dvp_cell_type_group_data.tsv.zip"
manifest_file = paths.manifest_dir / "frozen_inputs.json"

paths.require_file(data_file, "HPA DVP cell-type group archive")
paths.require_file(manifest_file, "frozen_inputs.json")
paths.require_file(notebook_path, "003d notebook source")

input_checksums = {
    "references/hpa/dvp_cell_type_group_data.tsv.zip": sha256_file(data_file),
    "frozen_inputs.json": sha256_file(manifest_file),
}

# Reserve the stage once, after preflight.
run_dir = paths.prepare_stage("03d_normal_tissue_protein")
df_dir = run_dir / "tables"
fig_dir = run_dir / "figures"

for directory in (df_dir, fig_dir):
    directory.mkdir(parents=True, exist_ok=True)

(run_dir / "input_checksums.json").write_text(
    json.dumps(input_checksums, indent=2) + "\n",
    encoding="utf-8",
)
write_run_provenance(
    paths,
    run_dir,
    notebook_path=notebook_path,
    input_checksums=input_checksums,
)

CalledProcessError: Command '['git', 'rev-parse', 'HEAD']' returned non-zero exit status 128.

In [ ]:
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42


In [ ]:
gene_order = ["ROR1", "EGFR", "ERBB2", "EPCAM"]


In [ ]:
organ_groups = {
    "Blood / Immune": ["b-cells", "t-cells", "macrophages", "neutrophils"],
    "Brain": ["neurons", "astrocytes and neuropil", "microglia and neuropil"],
    "Heart": ["cardiomyocytes", "capillaries"],
    "Kidney": ["glomeruli", "proximal tubules", "distal tubules", "collecting ducts"],
    "Liver": ["hepatocytes"],
    "Lung": ["alveolar cell types"],
    "Ovary": ["granulosa cells", "oocytes"],
    "Pancreas": ["pancreatic islets", "pancreas epithelial cells"],
    "Salivary gland": ["large ducts"],
    "Skin": ["keratinocytes"],
    "Skeletal muscle": ["skeletal myofibers"],
    "Fallopian tube": ["fallopian tube cells"],
    "Vasculature": ["smooth muscle cells"],
}


In [ ]:
def load_hpa(path):
    """Keep non-detection as NaN; the zero convention is applied only in scoring."""
    path = Path(path)
    with zipfile.ZipFile(path) as archive:
        members = [name for name in archive.namelist() if name.endswith(".tsv")]
        if len(members) != 1:
            raise ValueError("Expected exactly one TSV in the cached HPA archive.")
        with archive.open(members[0]) as handle:
            table = pd.read_csv(handle, sep="\t")
    targets = table.loc[table["Gene name"].isin(gene_order)].copy()
    if targets.duplicated(["Cell type name", "Gene name"]).any():
        raise ValueError(
            "Duplicate target/cell-type measurements need an explicit aggregation rule."
        )
    abundance = targets.pivot(
        index="Cell type name", columns="Gene name", values="Intensity"
    ).reindex(columns=gene_order)
    expected = [cell for members in organ_groups.values() for cell in members]
    if set(expected) != set(abundance.index):
        raise ValueError(
            f"HPA cell-type mapping changed: {set(expected) ^ set(abundance.index)}"
        )
    abundance = abundance.reindex(expected)
    if abundance.lt(0).any().any() or abundance.notna().sum().eq(0).any():
        raise ValueError("Invalid intensities or a wholly unquantified target.")
    provenance = {
        "file": path.name,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        "formula": "A=sqrt(I)/max(sqrt(I)); L=min(A_i,A_j); CTLI=sum_c L",
        "missing_policy": "zero contribution to the primary score; biological abundance remains unknown",
        "n_groups": len(abundance),
        "source_url": "https://www.proteinatlas.org/download/tsv/dvp_cell_type_group_data.tsv.zip",
        "source_version": "not encoded in cached archive; verify before deposit",
        "interpretation": "exploratory total-protein co-representation; not same-cell surface co-display or safety",
    }
    return (abundance, provenance)


In [ ]:
abundance, provenance = load_hpa(data_file)


In [ ]:
abundance


Non-quantified intensities remain missing in the source table and appear as crosses in the figure. They contribute zero only to the declared primary score. Non-detection does not establish absence. HPA pools cells within groups; this does not measure same-cell surface co-display.

Source: [HPA-DVP methods](https://www.proteinatlas.org/humanproteome/single%2Bcell/dvp/method).

In [ ]:
def score_hpa(abundance, transform="sqrt", missing_value=0.0):
    """Use one global denominator for all targets and retain the measured mask."""
    transforms = {"sqrt": np.sqrt, "linear": lambda x: x, "log1p": np.log1p}
    transformed = transforms[transform](abundance)
    denominator = float(np.nanmax(transformed.to_numpy()))
    if not np.isfinite(denominator) or denominator <= 0:
        raise ValueError("The common intensity scale must be positive and finite.")
    filled = transforms[transform](abundance.fillna(missing_value)) / denominator
    local = pd.DataFrame(
        {
            f"{a}+{b}": filled[[a, b]].min(axis=1)
            for a, b in itertools.combinations(gene_order, 2)
        }
    )
    observed = pd.DataFrame(
        {
            f"{a}+{b}": abundance[[a, b]].notna().all(axis=1)
            for a, b in itertools.combinations(gene_order, 2)
        }
    )
    return (transformed / denominator, local, observed)


In [ ]:
hpa_scaled, hpa_local, hpa_observed = score_hpa(abundance)


In [ ]:
hpa_summary = pd.DataFrame(
    {
        "ctli_sum": hpa_local.sum(),
        "ctli_max": hpa_local.max(),
        "top_cell_type": hpa_local.idxmax(),
        "n_joint_quantified": hpa_observed.sum(),
        "n_groups": len(hpa_local),
        "joint_quantified_mean": hpa_local.where(hpa_observed).mean(),
    }
).sort_values("ctli_sum")


In [ ]:
hpa_summary["top_share_of_sum"] = hpa_summary.ctli_max.div(
    hpa_summary.ctli_sum.replace(0, np.nan)
)


In [ ]:
hpa_summary["rank"] = hpa_summary.ctli_sum.rank(method="min").astype(int)


In [ ]:
hpa_transforms = []


In [ ]:
for hpa_transform in ("linear", "sqrt", "log1p"):
    hpa__, hpa_values, hpa__ = score_hpa(abundance, hpa_transform)
    for hpa_pair, hpa_total in hpa_values.sum().items():
        hpa_transforms.append(
            {
                "transform": hpa_transform,
                "pair": hpa_pair,
                "ctli_sum": hpa_total,
                "rank": hpa_values.sum().rank(method="min")[hpa_pair],
            }
        )


In [ ]:
hpa_omitted = []


In [ ]:
for hpa_cell_type in hpa_local.index:
    hpa_totals = hpa_local.drop(index=hpa_cell_type).sum()
    for hpa_pair, hpa_total in hpa_totals.items():
        hpa_omitted.append(
            {
                "dropped": hpa_cell_type,
                "pair": hpa_pair,
                "ctli_sum": hpa_total,
                "rank": hpa_totals.rank(method="min")[hpa_pair],
            }
        )


In [ ]:
hpa_positive = abundance.to_numpy()[
    np.isfinite(abundance.to_numpy()) & (abundance.to_numpy() > 0)
]


In [ ]:
hpa_missing_rows = []


In [ ]:
for hpa_name, hpa_fill in (
    ("zero contribution", 0.0),
    ("pooled detected q10", float(np.quantile(hpa_positive, 0.1))),
    ("pooled detected median", float(np.median(hpa_positive))),
):
    hpa__, hpa_values, hpa__ = score_hpa(abundance, missing_value=hpa_fill)
    hpa_totals = hpa_values.sum()
    for hpa_pair, hpa_total in hpa_totals.items():
        hpa_missing_rows.append(
            {
                "scenario": hpa_name,
                "assumed_missing_intensity": hpa_fill,
                "pair": hpa_pair,
                "ctli_sum": hpa_total,
                "rank": hpa_totals.rank(method="min")[hpa_pair],
            }
        )


In [ ]:
hpa_coverage = pd.DataFrame(
    {
        "n_quantified": abundance.notna().sum(),
        "n_unquantified": abundance.isna().sum(),
        "n_groups": len(abundance),
    }
)


In [ ]:
results = dict(
    scaled=hpa_scaled,
    local=hpa_local,
    observed=hpa_observed,
    summary=hpa_summary,
    coverage=hpa_coverage,
    transforms=pd.DataFrame(hpa_transforms),
    leave_one_out=pd.DataFrame(hpa_omitted),
    missing_scenarios=pd.DataFrame(hpa_missing_rows),
)


In [ ]:
results["coverage"]


In [ ]:
results["summary"]


In [ ]:
def export_hpa(abundance, results, provenance, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    abundance.to_csv(output_dir / "hpa_dvp_raw_intensities.csv")
    names = {
        "local": "hpa_dvp_coexpression_by_celltype",
        "summary": "hpa_dvp_liability_summary",
        "coverage": "hpa_dvp_target_coverage",
        "transforms": "hpa_dvp_transform_sensitivity",
        "leave_one_out": "hpa_dvp_leave_one_group_out",
        "missing_scenarios": "hpa_dvp_missingness_scenarios",
    }
    for key, name in names.items():
        results[key].to_csv(
            output_dir / f"{name}.csv", index=key in ("local", "summary", "coverage")
        )
    results["summary"].ctli_sum.rename("ctli").to_csv(
        output_dir / "hpa_dvp_liability_score.csv"
    )
    (output_dir / "hpa_dvp_input_provenance.json").write_text(
        json.dumps(provenance, indent=2) + "\n"
    )


In [ ]:
export_hpa(abundance, results, provenance, df_dir)


Sensitivity analyses retain the maximum local score and leave-one-group-out ranking. Linear and log1p transforms use the same global scaling convention. Missing-value scenarios are stress tests, not inferred detection limits or biological imputations.

In [ ]:
results["transforms"].pivot(index="pair", columns="transform", values="rank")


In [ ]:
results["leave_one_out"].groupby("pair")["rank"].agg(["min", "max"])


In [ ]:
results["missing_scenarios"].pivot(index="pair", columns="scenario", values="rank")


In [ ]:
gene_colors = {
    "ROR1": "#d62728",
    "EGFR": "#9467bd",
    "ERBB2": "#2ca02c",
    "EPCAM": "#1f77b4",
}


In [ ]:
def draw_hpa(abundance, results, output_dir):
    """One cohesive figure; missing measurements are visible as crosses."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    scaled, local, observed = (results[key] for key in ("scaled", "local", "observed"))
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(17, 12),
        gridspec_kw={"width_ratios": [1.5, 1.25, 1]},
        layout="constrained",
    )
    positions, labels, headers = ({}, [], [])
    cursor = 0
    for organ, members in organ_groups.items():
        headers.append((cursor, organ))
        labels.append("")
        cursor += 1
        for cell in members:
            positions[cell] = cursor
            labels.append(cell)
            cursor += 1
    for column, gene in enumerate(gene_order):
        for cell, y in positions.items():
            value = scaled.loc[cell, gene]
            if pd.isna(value):
                axes[0].scatter(
                    column, y, marker="x", s=14, color="#AAAAAA", linewidths=0.7
                )
            elif value > 0:
                axes[0].scatter(
                    column, y, s=400 * value, color=gene_colors[gene], edgecolors="none"
                )
    axes[0].set(
        xticks=range(4),
        xticklabels=gene_order,
        yticks=range(cursor),
        yticklabels=labels,
        ylim=(cursor - 0.3, -0.8),
        title="a  Measured total-protein intensity",
        xlabel="square-root intensity / common global maximum",
    )
    for y, organ in headers:
        axes[0].text(
            -0.57,
            y,
            organ,
            transform=axes[0].get_yaxis_transform(),
            fontweight="bold",
            va="center",
        )
    points = None
    for column, pair in enumerate(local):
        for cell, y in positions.items():
            if not observed.loc[cell, pair]:
                axes[1].scatter(
                    column, y, marker="x", s=12, color="#AAAAAA", linewidths=0.6
                )
            else:
                value = local.loc[cell, pair]
                points = axes[1].scatter(
                    column, y, s=400 * value, c=[value], cmap="viridis", vmin=0, vmax=1
                )
    axes[1].set(
        xticks=range(len(local.columns)),
        xticklabels=[x.replace("+", "–") for x in local],
        yticks=[],
        ylim=(cursor - 0.3, -0.8),
        title="b  Limiting-antigen score",
    )
    axes[1].tick_params(axis="x", rotation=55)
    fig.colorbar(
        points,
        ax=axes[1],
        orientation="horizontal",
        pad=0.09,
        fraction=0.035,
        label="minimum scaled intensity",
    )
    totals = results["summary"].ctli_sum.sort_values(ascending=False)
    axes[2].barh(
        range(len(totals)),
        totals,
        color=[
            "#1f77b4" if name == "ROR1+EGFR" else "#CCCCCC" for name in totals.index
        ],
    )
    axes[2].set(
        yticks=range(len(totals)),
        yticklabels=[x.replace("+", "–") for x in totals.index],
        xlabel="CTLI (zero contribution for unquantified values)",
        title="c  Exploratory co-representation",
    )
    axes[2].invert_yaxis()
    axes[2].set_xlim(0, totals.max() * 1.2)
    for y, value in enumerate(totals):
        axes[2].text(value + totals.max() * 0.025, y, f"{value:.2f}", va="center")
    for axis in axes:
        axis.spines[["top", "right"]].set_visible(False)
        axis.grid(False)
    size_handles = [
        Line2D(
            [],
            [],
            marker="o",
            linestyle="none",
            color="#777777",
            markersize=np.sqrt(400 * value),
            label=f"{value:.2f}",
        )
        for value in (0.25, 0.5, 0.75, 1.0)
    ]
    size_handles.append(
        Line2D(
            [],
            [],
            marker="x",
            linestyle="none",
            color="#AAAAAA",
            label="not quantified",
        )
    )
    axes[0].legend(
        handles=size_handles,
        title="scaled intensity (bubble area)",
        ncol=3,
        loc="upper left",
        bbox_to_anchor=(0, -0.06),
        frameon=False,
        fontsize=8,
    )
    fig.suptitle(
        "HPA-DVP: 24 pooled cell-type groups; no breast tissue; one donor\nTotal protein, not surface antigen density or a validated safety estimate",
        fontsize=11,
    )
    fig.savefig(output_dir / 'normal_tissue_protein_corepresentation.svg', dpi=300, bbox_inches="tight")
    fig.savefig(output_dir / 'normal_tissue_protein_corepresentation.pdf', dpi=300, bbox_inches="tight")
    plt.show()
    return fig

In [ ]:
figure = draw_hpa(abundance, results, fig_dir)


HPA-DVP contains no breast tissue and profiles a limited panel from one donor. These results cannot establish normal-breast liability, accessible surface density or clinical safety. Raw intensities from different proteins are not calibrated molecule counts. The archive checksum is recorded; its release and retrieval date still require confirmation before deposit.

In [ ]:
session_info.show()
